In [1]:
import sys
import os
from pathlib import Path
import pprint

import numpy as np
import torch

sys.path.append("../../_src/_002_readdata")
sys.path.append("../../_src/_003_ml_f")
#sys.path.append("../")

from camelsh import camelsh
from mflstm import MFLSTM


In [2]:
data_path = Path("C:/Users/mfjakows/Datasets/CAMELSH").resolve()
#data_path = Path("C:/Users/mfjakows/Downloads/tva/CAMELSH").resolve()
train_entity_path = Path("../Input/testing.txt").resolve()
#train_entity_path = Path("Input/emory_oak.txt").resolve()
#train_entity_path = Path("Input/local.txt").resolve()


emory_staid = "03540500"


In [ ]:
dynamic_input = {
    "1D": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
    #"1h": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
    "1h": ["CAPE"],
}
target = ["Q_camelsh_obs_norm"]
forcing = ["nldas_hourly"]
static_input = [
    "p_mean", "pet_mean", "aridity_index", "p_seasonality", "frac_snow",
    "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur",
    "ele_mt_sav", "slp_dg_uav", "ria_ha_usu", "run_mm_syr", "gwt_cm_sav",
    "cly_pc_uav", "slt_pc_uav", "snd_pc_uav", "kar_pc_use", "prm_pc_use",
    "pac_pc_use", "crp_pc_use", "for_pc_use", "urb_pc_use", "DRAIN_SQKM"
]
#training_period = ["1987-01-01 00:00:00", "2009-12-31 23:00:00"]
#validation_period = ["2010-01-01 00:00:00", "2015-12-31 23:00:00"]
#testing_period = ["2016-01-01 00:00:00", "2022-12-31 23:00:00"]
training_period = ["2020-01-01 00:00:00", "2022-12-31 23:00:00"]

lookback_window = 0
SMOKE_TEST = True
model_configuration = {
    "n_dynamic_channels_lstm": 10,
    "no_of_layers": 1,
    "seq_length": 365 * 24,
    "custom_freq_processing": {
        "1D": {"n_steps": 351, "freq_factor": 24},
        "1h": {"n_steps": (365 - 351) * 24, "freq_factor": 1},
    },
    "predict_last_n": 1,
    #"predict_last_n": (24*14 + (365-14) - 1),
    "unique_prediction_blocks": True,
    "dynamic_embeddings": True,
    "hidden_size": 32 if SMOKE_TEST else 128,
    "batch_size_training": 16 if SMOKE_TEST else 128,
    "batch_size_evaluation": 128 if SMOKE_TEST else 1024,
    "no_of_epochs": 10 if SMOKE_TEST else 30,
    "dropout_rate": 0.4,
    "learning_rate": {1: 5e-4, 10: 1e-4, 25: 1e-5},
    "set_forget_gate": 3,
    "validate_every": 1 if SMOKE_TEST else 4,
    "validate_n_random_basins": 1 if SMOKE_TEST else -1,
}

seed = 110
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}


In [4]:
dataset = camelsh(
    dynamic_input=dynamic_input,
    target=target,
    forcing=forcing,
    sequence_length=model_configuration["seq_length"],
    time_period=training_period,
    path_data=str(data_path),
    path_entities=str(train_entity_path),
    check_NaN=True,
    predict_last_n=model_configuration["predict_last_n"],
    static_input=static_input,
    custom_freq_processing=model_configuration["custom_freq_processing"],
    dynamic_embedding=model_configuration["dynamic_embeddings"],
    unique_prediction_blocks=model_configuration["unique_prediction_blocks"],
    lookback_window=lookback_window,
)

In [5]:
if model_configuration.get("custom_freq_processing") and not model_configuration.get("dynamic_embeddings"):
    print("bruh")

In [6]:
model_configuration["dynamic_input_size"] = {key : len(value) for key, value in dynamic_input.items()}

model_configuration["input_size_lstm"] = model_configuration["n_dynamic_channels_lstm"] + len(static_input)

model_configuration["predict_last_n"] = 1

device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


In [7]:
dataset.calculate_basin_std()
dataset.calculate_global_statistics()
dataset.standardize_data()


In [8]:
print(len(dataset))
dataset[0]

52023


{'x_d_1D': tensor([[-0.3404, -0.2586,  0.1316,  ..., -0.3812,  0.8151,  0.2167],
         [-0.3546, -0.2586, -1.4009,  ..., -1.2694, -0.1764, -0.7677],
         [-0.3546, -0.2586, -0.5369,  ..., -1.0024, -0.1655, -0.0405],
         ...,
         [-0.3480, -0.2586, -0.5585,  ..., -0.7702,  2.0218,  0.2742],
         [-0.3546, -0.2586, -0.7130,  ..., -1.2617,  1.0259, -0.6289],
         [-0.3479, -0.2586, -0.0936,  ..., -1.2597, -1.3371, -0.7733]]),
 'x_d_1h': tensor([[-0.3357],
         [-0.3452],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3546],
         [-0.3540],
         [-0.3533],
         [-0.3527],
         [-0.3106],
         [-0.2685],
         [-0.2264],
         [-0.2169],
         [-0.2074],
         [-0.1979],
         [-0.2125],
         [-0.2272],
         [-0.2418]

In [9]:
#import pprint
#did = 10000
##pprint.pprint((dataset[0]))
#pprint.pprint(len((dataset[0]['x_d_1D'])))
#pprint.pprint(len((dataset[0]['x_d_1h'])))

##pprint.pprint((dataset[did]))
#pprint.pprint(len((dataset[did]['x_d_1D'])))
#pprint.pprint(len((dataset[did]['x_d_1h'])))

##len(dataset)
#pprint.pprint(len((dataset[did])))
#pprint.pprint(dataset[did]['x_d_1D'])

In [10]:
#model = MFLSTM(
    #model_configuration)

In [11]:
import pandas as pd
from torch.utils.data import DataLoader

# 1) Inspect what one sample contains
s0 = dataset[0]
print(s0.keys())   # expect things like: x_d_1D, x_d_1h, x_s, y_obs, date, ...

# 2) Collect observed flow time series from dataset
loader = DataLoader(
    dataset=dataset,
    batch_size=512,
    shuffle=False,
    drop_last=False,
    collate_fn=dataset.collate_fn,
)

rows = []
for batch in loader:
    # y_obs shape is typically [B, predict_last_n, 1]
    y = batch["y_obs"].reshape(-1).cpu().numpy()
    t = pd.to_datetime(batch["date"].reshape(-1))
    rows.append(pd.DataFrame({"y_obs": y}, index=t))

df_obs = pd.concat(rows).sort_index()

# optional: remove duplicate timestamps if any overlap exists
df_obs = df_obs[~df_obs.index.duplicated(keep="last")]

display(df_obs.head())
display(df_obs.tail())
print("n_points:", len(df_obs))

dict_keys(['x_d_1D', 'x_d_1h', 'x_s', 'y_obs', 'basin_std', 'basin', 'date'])


,y_obs
2020-01-01 00:00:00,-0.375036
2020-01-01 01:00:00,1.803917
2020-01-01 02:00:00,-0.379513
2020-01-01 03:00:00,-0.383990
2020-01-01 04:00:00,1.626954


,y_obs
2022-12-31 19:00:00,0.091964
2022-12-31 20:00:00,0.101796
2022-12-31 21:00:00,0.116543
2022-12-31 22:00:00,-0.360113
2022-12-31 23:00:00,-0.313104


n_points: 26304


In [12]:
#df_fews = pd.read_csv("holston_smf.csv")
#df_fews = pd.read_csv("holston_smf.csv", parse_dates=["date"])
#df_fews = pd.read_csv("Emory_Oakdale_Flow_FEWS.csv")
df_fews = pd.read_csv("../holston_smf.csv", parse_dates=[0], header=None,
                      names=["date","flow"])
df_fews = df_fews[df_fews["date"] <= "2022-12-31 23:00:00"]
df_fews
df_obs_unnormalized = df_obs.copy()
df_obs_unnormalized["y_obs"] = df_obs_unnormalized["y_obs"].apply(lambda x: x * (341.9203 * 10**6) / (86400*1000))
df_obs_unnormalized

,y_obs
2020-01-01 00:00:00,-1.484172
2020-01-01 01:00:00,7.138841
2020-01-01 02:00:00,-1.501889
2020-01-01 03:00:00,-1.519606
2020-01-01 04:00:00,6.438527
...,...
2022-12-31 19:00:00,0.363941
2022-12-31 20:00:00,0.402847
2022-12-31 21:00:00,0.461207
2022-12-31 22:00:00,-1.425114


In [13]:
pprint.pprint((np.abs(df_fews["flow"].values - df_obs_unnormalized["y_obs"].values)).mean())

np.float64(6.302946261532078)


In [14]:
print(type(df_fews.iloc[0]["date"]))

<class 'pandas.Timestamp'>
